In [ ]:
import pandas as pd
import requests
import time
import os
import zipfile
import urllib.request
from tqdm import tqdm 
from dotenv import load_dotenv, find_dotenv


In [ ]:

# Tự động tìm file .env ở các thư mục cha
load_dotenv(find_dotenv())

# Sử dụng biến bình thường
TMDB_API_KEY = os.getenv("TMDB_API_KEY")

In [ ]:
DATA_PATH = "../data"

In [ ]:
# !pip install tqdm

In [ ]:
# !pip3 install kagglehub 

In [ ]:
# !pip install kagglehub[hf-datasets]

In [ ]:
import kagglehub


MOVIELENS_DIR = os.path.join(DATA_PATH, 'movielens')
path = ""
if not os.path.exists(MOVIELENS_DIR):
    # Download latest version
    path = kagglehub.dataset_download("grouplens/movielens-latest-full")
else:
    path = MOVIELENS_DIR
    
print("Path to dataset files:", path)

In [36]:
full_df = {
    "ratings": pd.read_csv(path + "/ratings.csv"),
    "tags": pd.read_csv(path + "/tags.csv"),
    "links": pd.read_csv(path + "/links.csv"),
    "movies": pd.read_csv(path + "/movies.csv"),
    "genome_scores": pd.read_csv(path + "/genome_scores.csv"),
    "genome_tags": pd.read_csv(path + "/genome_tags.csv")
}

In [ ]:
print('Số phim: ', full_df['movies'].shape[0]) 
print('Số người dùng: ', full_df['ratings']['userId'].nunique())
print('Số lượt đánh giá: ', full_df['ratings'].shape[0])

In [ ]:
for name, df in full_df.items():
    print(f"{50*'='}")
    print(f"DataFrame '{name}':")
    print(df.head())
    
    save_path = os.path.join(MOVIELENS_DIR, f"{name}.csv")
    df.to_csv(save_path, index=False)
    print("\n")

In [ ]:
# 1. Nhóm theo movieId để tính Số lượt đánh giá và Điểm trung bình
movie_stats = full_df['ratings'].groupby('movieId').agg(
    rating_count=('rating', 'count'),
    rating_mean=('rating', 'mean')
).reset_index()

# 2. Áp dụng tiêu chuẩn lọc (Filter)
MIN_RATINGS = 10  # Phim phải có ít nhất 50 người rate
MIN_SCORE = 2.0   # Điểm trung bình từ 3.0 trở lên

quality_movies = movie_stats[
    (movie_stats['rating_count'] >= MIN_RATINGS) & 
    (movie_stats['rating_mean'] >= MIN_SCORE)
]

print(f"Số phim ban đầu: {len(movie_stats)}")
print(f"Số phim 'Đáng xem' sau khi lọc: {len(quality_movies)}")

# 3. Gộp (Merge) kết quả đã lọc với bảng links_df để lấy tmdbId
filtered_links_df = pd.merge(quality_movies, full_df['links'], on='movieId', how='inner')

display(filtered_links_df.count())
display(filtered_links_df.sort_values(by='rating_count', ascending=False).head())

# lọc phim đã crawl từ TMDB
TMDB_DIR = os.path.join(DATA_PATH, 'tmdb')
tmdb_df = pd.read_csv(os.path.join(TMDB_DIR, 'movies_data_fully_translated.csv'))
display(tmdb_df.count())
display(tmdb_df.head(3))

# Lấy những phim chưa crawl từ TMDB
filtered_links_df = filtered_links_df[~filtered_links_df['tmdbId'].isin(tmdb_df['tmdbId'])]
print(f"Số phim đáng xem chưa crawl từ TMDB: {len(filtered_links_df)}")

In [37]:
MIN_RELEASE_YEAR = 1995

def fetch_movie_vietnamese_metadata(tmdb_id):
    """Hàm gọi API TMDB để lấy thông tin phim bằng tiếng Việt"""
    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}?api_key={TMDB_API_KEY}&language=vi-VN"
    
    try:
        response = requests.get(url, timeout=10)
        # Nếu request thành công
        if response.status_code == 200:
            data = response.json()
            
            # check date sau 1995
            # if 'release_date' in data and data['release_date']:
            #     release_year = int(data['release_date'].split('-')[0])
            #     if release_year < MIN_RELEASE_YEAR:
            #         return None
            is_fetch_again = False
            # Nếu không có tiếng việt, gọi lại với ngôn ngữ mặc định (tiếng Anh)
            if not data.get('overview') or data.get('overview') == '':
                url_en = f"https://api.themoviedb.org/3/movie/{tmdb_id}?api_key={TMDB_API_KEY}&language=en-US"
                response_en = requests.get(url_en, timeout=10)
                if response_en.status_code == 200:
                    data_en = response_en.json()
                    # data['title'] = data_en.get('title', '')
                    data['overview'] = data_en.get('overview', '')
                    is_fetch_again = True
            
            return {
                'tmdbId': tmdb_id,
                'title_vi': data.get('title', ''),
                'overview_vi' :is_fetch_again == False and data.get('overview', '') or '', # Nếu trống thì sẽ dịch ở giai đoạn sau
                # Lấy danh sách tên thể loại và nối thành chuỗi
                'genres': ", ".join([g['name'] for g in data.get('genres', [])]) if data.get('genres') else "",
                'release_date': data.get('release_date', ''),
                'poster_path': data.get('poster_path', '') # Rất hữu ích cho frontend sau này,
                
            } | data
        # Nếu phim không tồn tại trên TMDB
        elif response.status_code == 404:
            return None
    except Exception as e:
        print(f"Lỗi khi fetch tmdbId {tmdb_id}: {e}")
        return None
    return None

In [38]:
tmdb_id = 1760014 
metadata = fetch_movie_vietnamese_metadata(tmdb_id)
for k, v in metadata.items():
    print(f"{k}: {v}")


tmdbId: 1760014
title_vi: Fake
overview_vi: 
genres: [{'id': 53, 'name': 'Phim Gây Cấn'}, {'id': 35, 'name': 'Phim Hài'}]
release_date: 2026-08-29
poster_path: /Ogs0lTV4B0QfYe6nPPFCe4ULox.jpg
adult: False
backdrop_path: /cAHglSorAh1e8XwaSkhWe4Qm3rX.jpg
belongs_to_collection: None
budget: 1300
homepage: 
id: 1760014
imdb_id: None
origin_country: ['FR']
original_language: fr
original_title: Fake
overview: 
popularity: 1.885
production_companies: [{'id': 309270, 'logo_path': None, 'name': 'Nabû Studio', 'origin_country': ''}, {'id': 273238, 'logo_path': None, 'name': 'Université Toulouse-Jean Jaurès', 'origin_country': ''}]
production_countries: []
revenue: 0
runtime: 0
softcore: False
spoken_languages: []
status: Released
tagline: 
title: Fake
video: False
vote_average: 0.0
vote_count: 0


In [ ]:
movie_vi_data = []

# Nơi lưu phim
TMDB_DIR = os.path.join(DATA_PATH, 'tmdb')
if not os.path.exists(TMDB_DIR):
    os.makedirs(TMDB_DIR)
    
# Kiểm tra đã lưu trước hay chưa
if os.path.exists(os.path.join(TMDB_DIR, "movies_vietnamese_metadata.jsonl")):
    print("Dữ liệu phim tiếng Việt đã tồn tại, không cần fetch lại.")
    movie_vi_data = pd.read_json(os.path.join(TMDB_DIR, "movies_vietnamese_metadata.jsonl"), lines=True).to_dict(orient='records')




def fetch_movie_batch(limit=1000, offset=0):
    """Hàm fetch một batch phim từ filtered_links_df"""
    batch_links = filtered_links_df.iloc[offset:offset+limit]
    batch_data = [] 
    
    for index, row in tqdm(batch_links.iterrows(), total=batch_links.shape[0], desc=f"Đang fetch batch {offset + 1}-{offset + limit}"):
        movie_id = row['movieId']
        tmdb_id = row['tmdbId']
        
        metadata = fetch_movie_vietnamese_metadata(tmdb_id)
        
        if metadata:
            metadata['movieId'] = movie_id
            batch_data.append(metadata)
        
        time.sleep(0.05)  # Tránh bị block API
    
    movie_vi_data.extend(batch_data)
    save_movie(batch_data, offset, limit)
    return batch_data, offset + limit

def save_movie(data, offset, limit, prefix="movies_vietnamese_metadata", is_all=False):
    # Chuyển list các dictionary thành Pandas DataFrame
    movies_vi_df = pd.DataFrame(data)

    # Hiển thị vài dòng đầu để kiểm tra
    # display(movies_vi_df.head())

        
    start_index = offset + 1
    end_index = offset + len(data)
    if is_all:
        output_filename = os.path.join(TMDB_DIR, f"{prefix}.jsonl")
    else:
        output_filename = os.path.join(TMDB_DIR, f"{prefix}_{start_index}-{end_index}.jsonl")
    movies_vi_df.to_json(output_filename, orient='records', lines=True, force_ascii=False)
    print(f"\nĐã lưu thành công {len(movies_vi_df)} phim tiếng Việt vào file {output_filename}!")




In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def fetch_movie_batch_parallel(
    limit=1000,
    offset=0,
    max_workers=8,
    delay_seconds=0.0,
):
    """Fetch a movie batch concurrently and save it once all requests finish.

    max_workers controls the number of simultaneous TMDB requests.
    Use a small value such as 4-8 to reduce the risk of HTTP 429 responses.
    """
    batch_links = filtered_links_df.iloc[offset:offset + limit]

    def fetch_one(row):
        movie_id = row['movieId']
        tmdb_id = row['tmdbId']
        metadata = fetch_movie_vietnamese_metadata(tmdb_id)

        if delay_seconds > 0:
            time.sleep(delay_seconds)

        if metadata:
            metadata['movieId'] = movie_id
        return metadata

    batch_data = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(fetch_one, row)
            for _, row in batch_links.iterrows()
        ]
        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc=f"Đang fetch song song batch {offset + 1}-{offset + len(batch_links)}",
        ):
            metadata = future.result()
            if metadata:
                batch_data.append(metadata)

    movie_vi_data.extend(batch_data)
    # save_movie(batch_data, offset, limit)
    return batch_data, offset + len(batch_links)


# Ví dụ chạy từ offset hiện tại:
# max_workers = 8
# data, last_offset = fetch_movie_batch_parallel(
#     limit=batch_size,
#     offset=last_offset,
#     max_workers=max_workers,
# )

In [ ]:
max_workers = 8
batch_size = 2000
last_offset = 0

# Fetch toàn bộ dataset, kể cả batch cuối có ít hơn batch_size phim.
total_movies = len(filtered_links_df)
while last_offset < total_movies:
    data, last_offset = fetch_movie_batch_parallel(
        limit=batch_size,
        offset=last_offset,
        max_workers=max_workers,
    )
    print(f"Đã xử lý {min(last_offset, total_movies)}/{total_movies} phim")

In [ ]:
save_movie(movie_vi_data, offset=0, limit=len(movie_vi_data), prefix="movies_vietnamese_metadata_10-rating", is_all=True)

In [ ]:
movie_vi_data_df = pd.DataFrame(movie_vi_data)

In [ ]:
movie_vi_data_df.count()

In [ ]:
movie_vi_data_df.to_csv(os.path.join(TMDB_DIR, "movies_vietnamese_metadata_min-10-rating.csv"), index=False, encoding='utf-8-sig')

In [ ]:
movie_vi_data_df.info()

In [ ]:
movie_vi_data_df = movie_vi_data_df.dropna(subset=['overview_vi'])

# 2. Tính trung bình số từ (length) của cột overview_vi
mean_length = movie_vi_data_df['overview_vi'].str.split().str.len().mean()
print("Trung bình length của overview_vi:", mean_length)


In [ ]:
# 3. Lọc các phim có overview ngắn hơn 50 từ
short_movies_df = movie_vi_data_df[movie_vi_data_df['overview_vi'].str.split().str.len() == 0]
display(short_movies_df.sort_values('release_date', ascending=True).head(2000))



# Lọc phim năm 1995 trở đi
recent_movies_df = short_movies_df[short_movies_df['release_date'] >= '1990-01-01']
display(recent_movies_df.sort_values('release_date', ascending=True).head(10))
display(recent_movies_df.count())
recent_movies_df.to_csv(os.path.join(TMDB_DIR, "movies_vietnamese_metadata_min-10-rating_1990.csv"), index=False, encoding='utf-8-sig')

In [ ]:
recent_movies_df.count()
# short_movies_df.count()